# simulate_unit_root

> 对应代码：`EconometricsCode/code_in_notes/Chap.29/simulate_unit_root.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.29`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.29")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们构造 100 期的时间序列数据集，固定随机种子并声明 t 为时间变量，误差项 e 先置为缺失。

In [ ]:
%%stata
clear
set obs 100
set seed 250211
gen t = _n
tsset t
gen e = .

循环 15 次，每次重新抽取标准正态误差，并按 x_t = x_{t−1} + e_t 生成一条随机游走路径（起点为 0）。这样我们得到 15 条相互独立的单位根序列。

In [ ]:
%%stata
forvalues i=1/15{
	replace e = rnormal()
	gen x`i' = 0 if _n==1
	replace x`i' = L.x`i' + e if _n > 1
}

把 15 条随机游走画在同一张时间序列图上。可以看到各条路径呈现持续、无回归趋势的漂移——随机游走没有均值回复性，方差随时间线性增长，这正是非平稳性的直观表现。

In [ ]:
%%stata
tsline x*, legend(off)

画出第二条序列的样本自相关函数（ACF）。单位根过程的 ACF 衰减极其缓慢、近似线性下降，与平稳 AR(1) 的几何衰减形成鲜明对比，是识别单位根的重要图形证据。

In [ ]:
%%stata
ac x2